# 🧪 Notebook 05b — Evaluación semántica LLM de EMA Dialog2Flow · `version_4`

Esta notebook evalúa semántico-funcionalmente los vecinos recuperados por las variantes EMA.

Compara:

- `estatico`
- `acumulativo_uniforme`
- `ema_alpha_0_1` ... `ema_alpha_0_9`

sobre:

- `IVF`
- `HNSW`
- `IVFPQ`

Usa las recuperaciones top-100 persistidas por la notebook 04b y evalúa el top-10 con un LLM juez.

## 1️⃣ Instalación e imports

In [ ]:
!pip install -q openai pandas numpy tqdm scipy

In [ ]:
import os
import json
import time
from pathlib import Path
from getpass import getpass
from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display, Markdown

## 2️⃣ Montaje de Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3️⃣ Configuración

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/ANN/TF")
VERSION = "version_4"
SOURCE_VERSION = "version_3"

DATA_DIR = BASE_DIR / "data" / VERSION
RESULTS_DIR = BASE_DIR / "resultados" / VERSION

SOURCE_DATA_DIR = BASE_DIR / "data" / SOURCE_VERSION
SOURCE_EMBEDDINGS_DIR = SOURCE_DATA_DIR / "embeddings"

EMBEDDINGS_DIR = DATA_DIR / "embeddings"
SPLITS_DIR = DATA_DIR / "splits"
RETRIEVAL_DIR = DATA_DIR / "retrieval_results"

SEMANTIC_RESULTS_DIR = RESULTS_DIR / "semantic_llm"
SEMANTIC_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SHORT_NAME = "dialog2flow"
EMBEDDING_LABEL = "Dialog2Flow"
EMBEDDING_EXPERIMENTO = "dialog2flow-joint-bert-base"

RANDOM_STATE = 42
N_QUERIES_SPLIT = 10_000
N_QUERIES_LLM = 100
K_EVAL = 10
K_MAX = 100

CONTEXT_WINDOW = 2

INDICES_A_EVALUAR = ["IVF", "HNSW", "IVFPQ"]

ALPHA_VALUES = [round(x / 10, 1) for x in range(1, 10)]

VARIANTES_A_EVALUAR = (
    ["estatico", "acumulativo_uniforme"] +
    [f"ema_alpha_{str(alpha).replace('.', '_')}" for alpha in ALPHA_VALUES]
)

# Parámetros representativos usados para localizar retrieval_results
NPROBE_REPRESENTATIVO = 64
EF_SEARCH_REPRESENTATIVO = 256

OPENAI_MODEL = "gpt-4.1-mini"
TEMPERATURE = 0
SLEEP_BETWEEN_CALLS = 0.2

print("Variantes:", VARIANTES_A_EVALUAR)
print("Índices:", INDICES_A_EVALUAR)
print("N_QUERIES_LLM:", N_QUERIES_LLM)

## 4️⃣ Carga de datos textuales y split

In [ ]:
def find_existing_file(filename: str, candidates):
    for base in candidates:
        p = base / filename
        if p.exists():
            return p
    raise FileNotFoundError(
        f"No se encontró {filename}. Busqué en: " + ", ".join(str(c) for c in candidates)
    )


dialogs_path = find_existing_file(
    "dialogs-2.0.pkl",
    [DATA_DIR, EMBEDDINGS_DIR, SOURCE_EMBEDDINGS_DIR, SOURCE_DATA_DIR, BASE_DIR / "data"]
)

df = pd.read_pickle(dialogs_path)
print("DataFrame:", df.shape)
display(df.head())

split_query_path = SPLITS_DIR / f"query_idx_N{N_QUERIES_SPLIT}_seed{RANDOM_STATE}.npy"
split_index_path = SPLITS_DIR / f"index_idx_N{N_QUERIES_SPLIT}_seed{RANDOM_STATE}.npy"

if not split_query_path.exists() or not split_index_path.exists():
    raise FileNotFoundError(
        "Faltan los splits en version_4. Ejecutá antes notebook_04b o copiá/generá los archivos de split."
    )

query_idx_full = np.load(split_query_path)
index_idx = np.load(split_index_path)

rng = np.random.default_rng(RANDOM_STATE + 100)
query_sample = rng.choice(
    query_idx_full,
    size=min(N_QUERIES_LLM, len(query_idx_full)),
    replace=False,
).astype("int64")

print("Consultas split completo:", len(query_idx_full))
print("Consultas para LLM:", len(query_sample))
print("Primeras consultas:", query_sample[:10])

## 5️⃣ Funciones de contexto textual

In [ ]:
_df_ordered = df.reset_index().rename(columns={"index": "row_id"})
_dialogue_groups = {
    dialogue_id: g.sort_values("turn_id")["row_id"].tolist()
    for dialogue_id, g in _df_ordered.groupby("dialogue_id", sort=False)
}
_row_to_pos = {}
for dialogue_id, rows in _dialogue_groups.items():
    for pos, row_id in enumerate(rows):
        _row_to_pos[row_id] = pos


def get_context_rows(row_id: int, window: int = 2):
    dialogue_id = df.loc[row_id, "dialogue_id"]
    rows = _dialogue_groups[dialogue_id]
    pos = _row_to_pos[row_id]
    start = max(0, pos - window)
    return rows[start:pos + 1]


def format_turn(row_id: int) -> str:
    r = df.loc[row_id]
    speaker = r.get("speaker", "?")
    utterance = str(r.get("utterance", ""))
    return f"[{int(r.get('turn_id', -1))}] {speaker}: {utterance}"


def format_situation(row_id: int, window: int = 2) -> str:
    rows = get_context_rows(row_id, window=window)
    return "\n".join(format_turn(rid) for rid in rows)


def compact_metadata(row_id: int) -> dict:
    cols = ["dataset", "split", "dialogue_id", "turn_id", "speaker", "domains", "main_acts", "intents", "slots"]
    return {c: df.loc[row_id, c] for c in cols if c in df.columns}


print(format_situation(int(query_sample[0]), window=CONTEXT_WINDOW))

## 6️⃣ Carga de recuperaciones top-100 persistidas

In [ ]:
def retrieval_path(variante: str, index_type: str) -> Path:
    if index_type == "IVF":
        suffix = f"nprobe{NPROBE_REPRESENTATIVO}"
    elif index_type == "HNSW":
        suffix = f"efSearch{EF_SEARCH_REPRESENTATIVO}"
    elif index_type == "IVFPQ":
        suffix = f"nprobe{NPROBE_REPRESENTATIVO}"
    else:
        raise ValueError(index_type)

    return RETRIEVAL_DIR / (
        f"retrieval_{SHORT_NAME}_{variante}_{index_type}_{suffix}"
        f"_K{K_MAX}_N{N_QUERIES_SPLIT}_seed{RANDOM_STATE}.npz"
    )


def load_retrieval_records(variante: str, index_type: str) -> pd.DataFrame:
    path = retrieval_path(variante, index_type)
    if not path.exists():
        raise FileNotFoundError(f"No se encontró retrieval result: {path}")

    data = np.load(path, allow_pickle=True)
    query_rows_all = data["query_rows"].astype("int64")
    neighbor_rows_all = data["neighbor_rows"].astype("int64")
    distances_all = data["distances"].astype("float32")

    pos_by_query = {int(q): i for i, q in enumerate(query_rows_all)}

    records = []
    for q_order, q_row in enumerate(query_sample):
        q_row = int(q_row)
        if q_row not in pos_by_query:
            raise ValueError(f"La query {q_row} no aparece en {path.name}")

        pos = pos_by_query[q_row]

        for rank in range(K_EVAL):
            n_row = int(neighbor_rows_all[pos, rank])
            records.append({
                "embedding_name": EMBEDDING_EXPERIMENTO,
                "embedding_base": EMBEDDING_LABEL,
                "variant": variante,
                "index_type": index_type,
                "query_order": int(q_order),
                "query_row": q_row,
                "neighbor_rank": int(rank + 1),
                "neighbor_row": n_row,
                "distance": float(distances_all[pos, rank]),
                "query_utterance": str(df.loc[q_row, "utterance"]),
                "neighbor_utterance": str(df.loc[n_row, "utterance"]),
                "query_context": format_situation(q_row, window=CONTEXT_WINDOW),
                "neighbor_context": format_situation(n_row, window=CONTEXT_WINDOW),
            })

    return pd.DataFrame(records)


parts = []
for variante in tqdm(VARIANTES_A_EVALUAR, desc="Variantes"):
    for index_type in INDICES_A_EVALUAR:
        parts.append(load_retrieval_records(variante, index_type))

df_retrieval = pd.concat(parts, ignore_index=True)
print("Recuperaciones textuales:", df_retrieval.shape)
display(df_retrieval.head())

## 7️⃣ Inspección rápida

In [ ]:
def mostrar_vecinos(
    df_retrieval: pd.DataFrame,
    variante: str = "ema_alpha_0_5",
    index_type: str = "HNSW",
    query_order: int = 0,
    top_n: int = 10,
):
    sub = df_retrieval[
        (df_retrieval["variant"] == variante) &
        (df_retrieval["index_type"] == index_type) &
        (df_retrieval["query_order"] == query_order)
    ].sort_values("neighbor_rank").head(top_n)

    if sub.empty:
        print("No hay registros.")
        return

    first = sub.iloc[0]
    display(Markdown(f"### Consulta `{query_order}` — {variante} / {index_type}"))
    display(Markdown("**Situación de consulta:**"))
    print(first["query_context"])

    display(sub[["neighbor_rank", "distance", "neighbor_row", "neighbor_utterance"]])


mostrar_vecinos(df_retrieval, variante="ema_alpha_0_5", index_type="HNSW", query_order=0)

## 8️⃣ Cliente OpenAI

In [ ]:
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Ingresá OPENAI_API_KEY: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("Cliente OpenAI configurado.")

## 9️⃣ Prompt y esquema del juez

In [ ]:
EVALUATION_SCHEMA = {
    "name": "dialog_retrieval_evaluation",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "evaluations": {
                "type": "array",
                "minItems": K_EVAL,
                "maxItems": K_EVAL,
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "rank": {"type": "integer", "minimum": 1, "maximum": K_EVAL},
                        "semantic_similarity": {"type": "integer", "minimum": 1, "maximum": 5},
                        "functional_similarity": {"type": "integer", "minimum": 1, "maximum": 5},
                        "memory_usefulness": {"type": "integer", "minimum": 1, "maximum": 5},
                        "overall_similarity": {"type": "integer", "minimum": 1, "maximum": 5},
                        "brief_reason": {"type": "string"},
                    },
                    "required": [
                        "rank",
                        "semantic_similarity",
                        "functional_similarity",
                        "memory_usefulness",
                        "overall_similarity",
                        "brief_reason",
                    ],
                },
            }
        },
        "required": ["evaluations"],
    },
    "strict": True,
}

SYSTEM_PROMPT = """
You are an expert evaluator of task-oriented dialogue retrieval.
Your task is to judge whether retrieved dialogue situations are semantically and functionally similar to a query dialogue situation.
Focus on task-oriented dialogue behavior, not only lexical overlap.
Use the 1-5 scale consistently.
Return only valid JSON following the schema.
""".strip()


def build_judge_prompt(group: pd.DataFrame) -> str:
    group = group.sort_values("neighbor_rank")
    first = group.iloc[0]

    lines = []
    lines.append("Evaluate whether each retrieved neighbor is similar to the query situation.")
    lines.append("")
    lines.append("Scoring scale:")
    lines.append("1 = unrelated")
    lines.append("2 = weak or superficial relation")
    lines.append("3 = partial similarity")
    lines.append("4 = clear semantic/functional similarity")
    lines.append("5 = highly equivalent dialogue situations")
    lines.append("")
    lines.append("QUERY SITUATION:")
    lines.append(first["query_context"])
    lines.append("")
    lines.append("RETRIEVED NEIGHBORS:")

    for _, r in group.iterrows():
        lines.append("")
        lines.append(f"Neighbor rank {int(r['neighbor_rank'])}:")
        lines.append(r["neighbor_context"])

    lines.append("")
    lines.append("Return one evaluation object for each neighbor rank from 1 to 10.")
    return "\n".join(lines)


def evaluate_group_with_llm(group: pd.DataFrame, model: str = OPENAI_MODEL) -> dict:
    user_prompt = build_judge_prompt(group)

    response = client.chat.completions.create(
        model=model,
        temperature=TEMPERATURE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": EVALUATION_SCHEMA,
        },
    )

    content = response.choices[0].message.content
    return json.loads(content)

## 🔟 Evaluación con cache

In [ ]:
CACHE_PATH = SEMANTIC_RESULTS_DIR / "llm_semantic_scores_cache_ema_dialog2flow.jsonl"


def load_cache(path: Path) -> dict:
    cache = {}
    if not path.exists():
        return cache

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                cache[item["key"]] = item["result"]
    return cache


def append_cache(path: Path, key: str, result: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps({"key": key, "result": result}, ensure_ascii=False) + "\n")


cache = load_cache(CACHE_PATH)
print("Entradas en cache:", len(cache))


score_rows = []

group_cols = ["variant", "index_type", "query_order"]

for key_tuple, group in tqdm(df_retrieval.groupby(group_cols, sort=False), desc="Evaluando grupos"):
    variante, index_type, query_order = key_tuple
    cache_key = f"{variante}|{index_type}|{query_order}|k{K_EVAL}|model{OPENAI_MODEL}"

    if cache_key in cache:
        result = cache[cache_key]
    else:
        result = evaluate_group_with_llm(group)
        append_cache(CACHE_PATH, cache_key, result)
        cache[cache_key] = result
        time.sleep(SLEEP_BETWEEN_CALLS)

    evals = result["evaluations"]

    eval_by_rank = {int(e["rank"]): e for e in evals}

    for _, r in group.sort_values("neighbor_rank").iterrows():
        rank = int(r["neighbor_rank"])
        e = eval_by_rank[rank]

        score_rows.append({
            **r.to_dict(),
            "semantic_similarity": int(e["semantic_similarity"]),
            "functional_similarity": int(e["functional_similarity"]),
            "memory_usefulness": int(e["memory_usefulness"]),
            "overall_similarity": int(e["overall_similarity"]),
            "brief_reason": str(e.get("brief_reason", "")),
            "openai_model": OPENAI_MODEL,
            "temperature": TEMPERATURE,
        })

df_scores = pd.DataFrame(score_rows)
print("Scores:", df_scores.shape)
display(df_scores.head())

## 1️⃣1️⃣ Métricas MSS@10

In [ ]:
query_metrics = (
    df_scores
    .groupby(["embedding_base", "variant", "index_type", "query_order"], as_index=False)
    .agg(
        semantic_score_at10=("semantic_similarity", "mean"),
        functional_score_at10=("functional_similarity", "mean"),
        memory_score_at10=("memory_usefulness", "mean"),
        mss_at10=("overall_similarity", "mean"),
        query_row=("query_row", "first"),
    )
)

summary_metrics = (
    query_metrics
    .groupby(["embedding_base", "variant", "index_type"], as_index=False)
    .agg(
        n_queries=("query_order", "nunique"),
        mean_semantic_score_global_at10=("mss_at10", "mean"),
        sd_semantic_score_global_at10=("mss_at10", "std"),
        mean_semantic_score_at10=("semantic_score_at10", "mean"),
        sd_semantic_score_at10=("semantic_score_at10", "std"),
        mean_functional_score_at10=("functional_score_at10", "mean"),
        sd_functional_score_at10=("functional_score_at10", "std"),
        mean_memory_score_at10=("memory_score_at10", "mean"),
        sd_memory_score_at10=("memory_score_at10", "std"),
    )
)

summary_metrics["se_semantic_score_global_at10"] = (
    summary_metrics["sd_semantic_score_global_at10"] / np.sqrt(summary_metrics["n_queries"])
)
summary_metrics["ci95_semantic_score_global_at10"] = 1.96 * summary_metrics["se_semantic_score_global_at10"]

# Alpha explícito
def variant_alpha(v):
    if str(v).startswith("ema_alpha_"):
        return float(str(v).replace("ema_alpha_", "").replace("_", "."))
    return np.nan

summary_metrics["alpha"] = summary_metrics["variant"].map(variant_alpha)

for col in summary_metrics.select_dtypes(include=["float"]).columns:
    summary_metrics[col] = summary_metrics[col].round(4)

display(summary_metrics)

## 1️⃣2️⃣ Comparaciones contra estático y acumulativo uniforme

In [ ]:
# Pivot por query para permitir comparación pareada.
paired = query_metrics.pivot_table(
    index=["embedding_base", "index_type", "query_order"],
    columns="variant",
    values="mss_at10",
).reset_index()

baseline_static = "estatico"
baseline_acc = "acumulativo_uniforme"

rows = []
variants_to_compare = [v for v in VARIANTES_A_EVALUAR if v not in [baseline_static]]

try:
    from scipy.stats import wilcoxon
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

for index_type, group in paired.groupby("index_type", sort=False):
    for variant in variants_to_compare:
        if variant not in group.columns or baseline_static not in group.columns:
            continue

        deltas_static = group[variant] - group[baseline_static]

        if baseline_acc in group.columns and variant != baseline_acc:
            deltas_acc = group[variant] - group[baseline_acc]
        else:
            deltas_acc = pd.Series([np.nan] * len(group))

        if SCIPY_AVAILABLE and deltas_static.notna().sum() > 0 and not np.allclose(deltas_static.dropna(), 0):
            try:
                stat_static, p_static = wilcoxon(deltas_static.dropna())
            except Exception:
                stat_static, p_static = np.nan, np.nan
        else:
            stat_static, p_static = np.nan, np.nan

        if SCIPY_AVAILABLE and deltas_acc.notna().sum() > 0 and not np.allclose(deltas_acc.dropna(), 0):
            try:
                stat_acc, p_acc = wilcoxon(deltas_acc.dropna())
            except Exception:
                stat_acc, p_acc = np.nan, np.nan
        else:
            stat_acc, p_acc = np.nan, np.nan

        rows.append({
            "embedding_base": EMBEDDING_LABEL,
            "index_type": index_type,
            "variant": variant,
            "alpha": variant_alpha(variant),
            "n_queries": int(group["query_order"].nunique()),
            "mean_static": float(group[baseline_static].mean()),
            "mean_variant": float(group[variant].mean()),
            "mean_delta_vs_static": float(deltas_static.mean()),
            "median_delta_vs_static": float(deltas_static.median()),
            "pct_variant_better_than_static": float((deltas_static > 1e-9).mean()),
            "wilcoxon_p_vs_static": p_static,
            "mean_accumulative_uniform": float(group[baseline_acc].mean()) if baseline_acc in group.columns else np.nan,
            "mean_delta_vs_accumulative_uniform": float(deltas_acc.mean()) if deltas_acc.notna().any() else np.nan,
            "pct_variant_better_than_accumulative_uniform": float((deltas_acc > 1e-9).mean()) if deltas_acc.notna().any() else np.nan,
            "wilcoxon_p_vs_accumulative_uniform": p_acc,
        })

comparison_summary = pd.DataFrame(rows)

for col in comparison_summary.select_dtypes(include=["float"]).columns:
    comparison_summary[col] = comparison_summary[col].round(4)

display(comparison_summary)

## 1️⃣3️⃣ Tablas pivot para análisis alpha

In [ ]:
pivot_mss_alpha = summary_metrics.pivot_table(
    index="variant",
    columns="index_type",
    values="mean_semantic_score_global_at10",
)

pivot_sd_alpha = summary_metrics.pivot_table(
    index="variant",
    columns="index_type",
    values="sd_semantic_score_global_at10",
)

variant_order = ["estatico", "acumulativo_uniforme"] + [f"ema_alpha_{str(a).replace('.', '_')}" for a in ALPHA_VALUES]
pivot_mss_alpha = pivot_mss_alpha.reindex([v for v in variant_order if v in pivot_mss_alpha.index])
pivot_sd_alpha = pivot_sd_alpha.reindex([v for v in variant_order if v in pivot_sd_alpha.index])

print("MSS@10 medio por variante e índice")
display(pivot_mss_alpha)

print("SD de SS@10(q) por variante e índice")
display(pivot_sd_alpha)

## 1️⃣4️⃣ Guardado

In [ ]:
timestamp = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).strftime("%Y%m%d_%H%M%S_ARG")

paths = {
    "retrieval": SEMANTIC_RESULTS_DIR / f"retrieval_top10_textual_ema_dialog2flow_{timestamp}.csv",
    "scores": SEMANTIC_RESULTS_DIR / f"llm_semantic_scores_pairs_ema_dialog2flow_{timestamp}.csv",
    "query_metrics": SEMANTIC_RESULTS_DIR / f"llm_semantic_query_metrics_ema_dialog2flow_{timestamp}.csv",
    "summary": SEMANTIC_RESULTS_DIR / f"llm_semantic_summary_ema_dialog2flow_{timestamp}.csv",
    "comparison": SEMANTIC_RESULTS_DIR / f"ema_comparison_vs_baselines_dialog2flow_{timestamp}.csv",
    "pivot_mss": SEMANTIC_RESULTS_DIR / f"table_mss_at10_ema_alpha_pivot_{timestamp}.csv",
    "pivot_sd": SEMANTIC_RESULTS_DIR / f"table_mss_at10_ema_alpha_sd_pivot_{timestamp}.csv",
}

df_retrieval.to_csv(paths["retrieval"], index=False)
df_scores.to_csv(paths["scores"], index=False)
query_metrics.to_csv(paths["query_metrics"], index=False)
summary_metrics.to_csv(paths["summary"], index=False)
comparison_summary.to_csv(paths["comparison"], index=False)
pivot_mss_alpha.to_csv(paths["pivot_mss"])
pivot_sd_alpha.to_csv(paths["pivot_sd"])

# Latest
df_retrieval.to_csv(SEMANTIC_RESULTS_DIR / "retrieval_top10_textual_ema_dialog2flow_latest.csv", index=False)
df_scores.to_csv(SEMANTIC_RESULTS_DIR / "llm_semantic_scores_pairs_ema_dialog2flow_latest.csv", index=False)
query_metrics.to_csv(SEMANTIC_RESULTS_DIR / "llm_semantic_query_metrics_ema_dialog2flow_latest.csv", index=False)
summary_metrics.to_csv(SEMANTIC_RESULTS_DIR / "llm_semantic_summary_ema_dialog2flow_latest.csv", index=False)
comparison_summary.to_csv(SEMANTIC_RESULTS_DIR / "ema_comparison_vs_baselines_dialog2flow_latest.csv", index=False)
pivot_mss_alpha.to_csv(SEMANTIC_RESULTS_DIR / "table_mss_at10_ema_alpha_pivot_latest.csv")
pivot_sd_alpha.to_csv(SEMANTIC_RESULTS_DIR / "table_mss_at10_ema_alpha_sd_pivot_latest.csv")

print("Archivos guardados:")
for k, p in paths.items():
    print(k, "->", p)
print("Cache:", CACHE_PATH)